<a href="https://colab.research.google.com/github/miftahulhdd/BigData26_B_2411533005_MiftahulHudaAmri/blob/main/praktikum02/BD_B_P02_2411533005_MiftahulHudaAmri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import,install, dll

In [28]:
!pip install faker -q

In [29]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [30]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA = "/content/data"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_bersih.csv']


buat dataset sintetis

In [31]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []

for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [
        tgl.strftime("%Y-%m-%d"),
        tgl.strftime("%d/%m/%Y"),
        tgl.strftime("%d-%m-%Y")
    ]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)

    if random.random() < 0.3:
        metode = metode.lower()

    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [
    ("customer_name", 0.02),
    ("shipping_city", 0.03),
    ("payment_method", 0.015)
]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)

# Acak urutan baris
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Simpan data mentah
df.to_csv("transaksi_mentah.csv", index=False)

print("Jumlah baris:", len(df))

Jumlah baris: 515


In [32]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [33]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print("Jumlah baris setelah penanganan missing vlue:", len(df))

Jumlah baris setelah penanganan missing vlue: 495


In [34]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df["transaction_id"].duplicated().sum())

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5


In [35]:
df = df.drop_duplicates()

print("Jumlah baris setelah drop_duplicates():", len(df))

Jumlah baris setelah drop_duplicates(): 490


koreksi tipe data dan standardisasi format

In [36]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [37]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")

    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [38]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue

    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [39]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [40]:
df.to_csv("transaksi_bersih.csv", index=False)

print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [41]:
print(df.info())
print(df.head())
print("Jumlah baris akhir:", len(df))

<class 'pandas.core.frame.DataFrame'>
Index: 490 entries, 0 to 514
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    490 non-null    object 
 1   customer_name     490 non-null    object 
 2   product_name      490 non-null    object 
 3   category          490 non-null    string 
 4   price             490 non-null    float64
 5   quantity          490 non-null    int64  
 6   payment_method    490 non-null    string 
 7   transaction_date  490 non-null    object 
 8   shipping_city     490 non-null    string 
 9   rating            333 non-null    float64
dtypes: float64(2), int64(1), object(4), string(3)
memory usage: 42.1+ KB
None
  transaction_id           customer_name product_name      category     price  \
0       TRX00305         Ophelia Hartati    Quo Basic          Buku   50000.0   
1       TRX00500      Cut Maya Wijayanti     Delectus    Elektronik  500000.0   
2       TRX004

In [42]:
# from google.colab import drive
# import os

# drive.mount("/content/drive")

# folder_drive = "/content/drive/MyDrive/BigData/Praktikum2"
# os.makedirs(folder_drive, exist_ok=True)

# path_drive = folder_drive + "/transaksi_bersih.csv"
# df.to_csv(path_drive, index=False)

Studi Kasus

1.	Mengapa Angka 515 dan 490 Bisa Berbeda?

    Perbedaan ini bukan kesalahan sistem, melainkan hasil dari tahapan proses data yang berbeda antara tim IT dan tim Finance. Tim IT menarik data mentah (raw data) langsung dari sumber/database transaksi, sehingga jumlahnya masih 515 baris, termasuk seluruh data yang belum tervalidasi. Sementara itu, angka 490 baris yang dipakai tim Finance adalah data yang sudah melalui proses pembersihan (pre-processing).
    Selama proses tersebut, 25 baris dihapus secara sengaja karena kualitasnya tidak layak dipakai untuk pelaporan, yaitu:

  a.	5 baris data duplikat, transaksi yang tercatat lebih dari satu kali dengan kombinasi data yang identik, dihapus menggunakan drop_duplicates() agar tidak terjadi penghitungan ganda.
  
  b.	20 baris dengan data wajib yang kosong, transaksi tanpa informasi nama pelanggan (customer_name) atau metode pembayaran (payment_method), yang tidak bisa dianggap sebagai transaksi yang sah/lengkap secara akuntansi.
  
  Jadi, selisih 25 baris tersebut adalah data yang memang seharusnya tidak ikut dihitung, bukan data yang hilang begitu saja.

2.	Apakah 490 Baris “Lebih Benar” Dibanding 515 Baris?
    
    Ya, angka 490 baris lebih benar dan itulah yang seharusnya menjadi acuan laporan resmi. Hal ini berkaitan langsung dengan konsep Veracity dalam pengelolaan data (salah satu dimensi dalam Big Data selain Volume, Velocity, dan Variety). Veracity mengukur seberapa akurat dan dapat dipercaya suatu data untuk dijadikan dasar pengambilan Keputusan, bukan sekadar seberapa besar jumlahnya.
    Jika kita tetap memakai 515 baris, maka laporan Finance akan ikut menghitung:

  a.	Transaksi yang tercatat dua kali (duplikat) yang akan menggelembungkan (overstate) angka penjualan secara tidak wajar.
  
  b.	Transaksi yang tidak jelas identitas pembeli atau metode pembayarannya yang berisiko sulit ditelusuri dan divalidasi secara keuangan.

  Dengan kata lain, volume data yang besar tidak ada artinya jika kualitasnya (Veracity) rendah, inilah prinsip garbage in, garbage out: jika data kotor yang dimasukkan, maka laporan yang dihasilkan pun akan menyesatkan. Angka 490 baris, meski lebih kecil, justru mencerminkan transaksi yang valid dan dapat dipertanggungjawabkan.

3.	Mengapa Kolom Rating Dibiarkan Tetap Memiliki Missing Value?

    Kolom rating sengaja tidak diisi atau diimputasi, karena pemberian rating adalah tindakan opsional dari pembeli, tidak semua pembeli meluangkan waktu untuk memberi penilaian setelah bertransaksi. Ini berbeda sifatnya dengan kolom seperti shipping_city, yang memang seharusnya selalu terisi karena merupakan bagian wajib dari proses pengiriman.
    
    Jika nilai kosong pada kolom rating dipaksa diisi dengan angka tebakan atau asumsi tertentu, hal ini justru akan mendistorsi hasil analisis, misalnya membuat rata-rata rating terlihat lebih tinggi atau rendah dari kondisi sebenarnya, padahal angka tersebut bukan penilaian asli dari pembeli.
    Untuk menjawab kebutuhan tim Finance terkait “rating rata-rata semua transaksi”, pendekatan yang tepat adalah:
  
  a.	Menghitung rata-rata hanya dari transaksi yang benar-benar memiliki nilai rating (diberikan secara eksplisit oleh pembeli).
  
  b.	Data kosong pada kolom rating dikecualikan dari perhitungan tersebut, bukan dianggap sebagai nilai nol atau rata-rata.

Latihan

Latihan 1: Eksperimen Perubahan SEED menjadi 7

Kita akan menjalankan ulang pipeline data menggunakan SEED = 7 untuk membandingkan jumlah baris transaksi mentah dan bersih dengan hasil saat SEED = 42.

In [43]:
import numpy as np
import pandas as pd
from faker import Faker
import random

# Pipeline Menggunakan SEED = 7
SEED_BARU = 7
np.random.seed(SEED_BARU)
random.seed(SEED_BARU)
fake = Faker("id_ID")
Faker.seed(SEED_BARU)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows_7 = []

for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [
        tgl.strftime("%Y-%m-%d"),
        tgl.strftime("%d/%m/%Y"),
        tgl.strftime("%d-%m-%Y")
    ]
    tanggal = random.choice(tgl_variants)
    metode = random.choice(metode_bayar)

    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows_7.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df_7 = pd.DataFrame(rows_7)

# Suntikkan missing value
for col, frac in [
    ("customer_name", 0.02),
    ("shipping_city", 0.03),
    ("payment_method", 0.015)
]:
    idx = df_7.sample(frac=frac, random_state=SEED_BARU).index
    df_7.loc[idx, col] = np.nan

# Duplikasi 15 baris
dup_rows_7 = df_7.sample(n=15, random_state=SEED_BARU)
df_7 = pd.concat([df_7, dup_rows_7], ignore_index=True)
df_7 = df_7.sample(frac=1, random_state=SEED_BARU).reset_index(drop=True)

# Simpan data mentah
df_7.to_csv("transaksi_mentah_seed7.csv", index=False)
baris_mentah_7 = len(df_7)

# dropna
df_7_clean = df_7.dropna(subset=["customer_name", "payment_method"]).copy()
df_7_clean["shipping_city"] = df_7_clean["shipping_city"].fillna("Tidak Diketahui")

# drop_duplicates
df_7_clean = df_7_clean.drop_duplicates()

# Simpan data bersih
df_7_clean.to_csv("transaksi_bersih_seed7.csv", index=False)
baris_bersih_7 = len(df_7_clean)

print("[SEED = 42] Mentah: 515 baris, Bersih: 490 baris")
print(f"[SEED = 7] Mentah: {baris_mentah_7} baris, Bersih: {baris_bersih_7} baris")

[SEED = 42] Mentah: 515 baris, Bersih: 490 baris
[SEED = 7] Mentah: 515 baris, Bersih: 490 baris


Latihan 2: Menambahkan Kolom is_valid_price

Kita akan menambahkan kolom boolean is_valid_price untuk mengonfirmasi validitas nilai harga pada data bersih (menggunakan data df hasil SEED = 42).

In [44]:
# Menambahkan kolom validasi harga
df["is_valid_price"] = df["price"] > 0

# Memeriksa apakah ada harga yang tidak valid (is_valid_price == False)
jumlah_tidak_valid = (df["is_valid_price"] == False).sum()

print(f"Jumlah transaksi dengan harga tidak valid: {jumlah_tidak_valid} baris.")
display(df[["transaction_id", "price", "is_valid_price"]].head())

Jumlah transaksi dengan harga tidak valid: 0 baris.


,transaction_id,price,is_valid_price
0,TRX00305,50000.0,True
1,TRX00500,500000.0,True
2,TRX00442,25000.0,True
3,TRX00154,250000.0,True
5,TRX00132,250000.0,True


Latihan 3: Menghitung Transaksi per Kategori

Menghitung sebaran frekuensi transaksi berdasarkan kolom category pada dataset bersih yang telah distandardisasi.

In [45]:
# Menghitung jumlah transaksi per category
sebaran_kategori = df["category"].value_counts()

print("Sebaran Jumlah Transaksi Per Kategori Produk:")
display(sebaran_kategori)

Sebaran Jumlah Transaksi Per Kategori Produk:


,count
category,
Olahraga,97
Kesehatan,91
Elektronik,89
Buku,82
Fashion,66
Rumah Tangga,65
